In [1]:
# ==========================================
# E-COMMERCE USER CONVERSION PREDICTION
# USING SUPPORT VECTOR MACHINE (SVM)
# ==========================================

# Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv("adtech_ecommerce_dataset.csv")

print("Dataset Shape:", df.shape)
print(df.head())

# ==========================================
# DEFINE TARGET VARIABLE
# ==========================================

target = "conversion"

X = df.drop(columns=[target])
y = df[target]

# ==========================================
# IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ==========================================

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

# ==========================================
# PREPROCESSING
# ==========================================

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ==========================================
# BUILD SVM PIPELINE
# ==========================================

svm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", SVC())
    ]
)

# ==========================================
# HYPERPARAMETER TUNING
# ==========================================

param_grid = {
    "classifier__kernel": ["rbf"],
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__gamma": [0.01, 0.1, 1]
}

grid_search = GridSearchCV(
    svm_pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

print("\nTraining SVM...")

grid_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid_search.best_params_)

# ==========================================
# PREDICTIONS
# ==========================================

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

# ==========================================
# MODEL EVALUATION
# ==========================================

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", round(accuracy, 4))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

# ==========================================
# BUSINESS INSIGHTS
# ==========================================

print("\n===================================")
print("BUSINESS INTERPRETATION")
print("===================================")

print("""
1. Users predicted as 1 are likely converters.

2. Marketing teams can target these users
   with personalized campaigns.

3. High-conversion users may receive:
      - Product recommendations
      - Loyalty rewards
      - Upsell offers

4. Predicted non-converters can receive:
      - Discount coupons
      - Remarketing ads
      - Cart abandonment reminders

5. SVM helps separate converter and
   non-converter behavior based on:
      - Click activity
      - Session duration
      - Cart additions
      - Purchase history
      - Traffic source
      - Device type
""")

Dataset Shape: (10000, 9)
   session_duration_sec  pages_viewed  clicks  ad_impressions device_type  \
0                  3184             3      37             473     Desktop   
1                  3517            12       8             372     Desktop   
2                   870             7      73             376      Mobile   
3                  1304            26      83              41      Mobile   
4                  1140            33      77              37      Tablet   

  traffic_source  cart_additions  past_purchases  conversion  
0      GoogleAds               6              30           1  
1       Facebook               8               9           1  
2        Organic               0              48           0  
3       Facebook               4              40           1  
4      GoogleAds              14              41           1  

Numerical Features:
['session_duration_sec', 'pages_viewed', 'clicks', 'ad_impressions', 'cart_additions', 'past_purchases']

Catego